In [ ]:
%run fabric-notebook-config

In [ ]:
def create_new_domain(display_name_in, description_in, parent_domain_in):
    '''
    TODO WARNING - THE UPDATE DOMAIN API IS BEING REVISED AND THIS MAY BREAK
    '''

    # requires Fabric Admin rights to successfully call the workspace membership update components
    # encapsulated Semantic Labs function so that we can do the Domain provisioning and also auto-assign Fabric administrative group to the domian
    with labs.service_principal_authentication(
        key_vault_uri = key_vault_uri,
        key_vault_tenant_id = 'tenant-fabric-sp',
        key_vault_client_id = 'fabric-admin-sp-app-id', 
        key_vault_client_secret = admin_sp_client_secret_name[1]
        ):
        labs.admin.create_domain(display_name_in, description_in, parent_domain_in)
        # create function doesn't return anything, so get the domain ID back
        new_domain_id = labs.admin.resolve_domain_id(display_name_in)
        # apply standard policy about who can assign stuff to the Domain - have to re-pass the desc
        labs.admin.update_domain(new_domain_id, description_in, 'AdminsOnly')

    # define a dictionary of the key identifiers so they can be passed to related / upstream functions
    new_domain_details = {
        'new_domain_id': new_domain_id,
        'new_domain_name': display_name_in,
        'new_domain_desc': description_in,
        'new_domain_parent': parent_domain_in
    }
    # confirm successful execution
    print("Created new Domain: " + display_name_in + " as " + new_domain_id)
    print("Go enter the domain as created into the config file's master copy, so that it's not detected an empty candiate for cleanup")
    return new_domain_details

In [ ]:
# test the module
try:
    create_new_domain('auto-test', "automated function testing", None)
    with labs.service_principal_authentication(
        key_vault_uri = key_vault_uri,
        key_vault_tenant_id = 'tenant-fabric-sp',
        key_vault_client_id = 'fabric-admin-sp-app-id', 
        key_vault_client_secret = admin_sp_client_secret_name[1]
        ):
        
        labs.admin.delete_domain('auto-test')
except:
    notebook_config_auth_failure_dict = {
         "message": "Function test failed: Could not Create & Delete domain"
    }
    mssparkutils.notebook.exit(notebook_config_auth_failure_dict)